# F1 INSIGHT — 03 · Data Mining

Runs the same four data-mining tasks exposed by `/api/mining/*`, directly
against the warehouse so every intermediate is inspectable.

* **Clustering** — K-Means over per-driver performance features, auto-k
* **Prediction** — race-winner classification using pre-race features only
* **Association rules** — Apriori over discretised race characteristics
* **Classification** — composite E/S/A/P performance scoring

> Requires a populated warehouse (see notebook 01 / `docs/setup.md`).

## Setup

In [ ]:
# Convenience setup: make the backend package importable (run from repo root).
import os, sys
ROOT = os.getcwd()
for candidate in (os.path.join(ROOT, "backend"), os.path.join(ROOT, "..", "backend")):
    if os.path.isdir(candidate):
        sys.path.insert(0, candidate)
        break
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)


from sqlalchemy import text
from app.db import SessionLocal


def q(sql, **params):
    "Run SQL against the warehouse and return rows as dicts."
    with SessionLocal() as db:
        rows = db.execute(text(sql), params).mappings().all()
    return [dict(r) for r in rows]


def counts():
    tables = ["dim_date", "dim_driver", "dim_constructor", "dim_circuit", "dim_race",
              "fact_race_result", "fact_qualifying", "fact_lap", "fact_pit_stop",
              "fact_driver_standing", "fact_constructor_standing"]
    out = []
    total = 0
    for t in tables:
        n = q(f"SELECT COUNT(*) AS n FROM {t}")[0]["n"]
        total += n
        out.append((t, n))
    out.append(("TOTAL", total))
    return out

## 1. K-Means clustering (auto k)

Drivers are described by 8 features (avg finish, avg qualifying, win rate,
podium rate, points/race, DNF rate, avg lap time, avg position gain). Features
are standardised, k is chosen by elbow + silhouette, and the response includes
cluster profiles: which features deviate most from the overall mean.

In [ ]:
from app.mining.clustering import run_clustering

with SessionLocal() as db:
    cl = run_clustering(db)

print('k =', cl['k'], '| selection:', cl['k_selection'])
print('silhouette =', cl['silhouette'], '| drivers =', cl['sample_size'])
print('pca explained variance =', cl['pca_explained_variance'])
for c in cl['clusters']:
    prof = next(p for p in cl['profile'] if p['cluster'] == c['cluster'])
    name = c['drivers'][0]['name']
    print('  cluster', c['cluster'], '| n =', c['size'],
          '| top =', name, '| standout =', prof['standout_features'])

Inspect the per-cluster centroids in detail:

In [ ]:
pd.DataFrame([
    {'cluster': c['cluster'], 'size': c['size'],
     **{f: round(v, 3) for f, v in c['centroid'].items()}}
    for c in cl['clusters']
])

## 2. Winner prediction (pre-race features only)

Trains Random Forest / Decision Tree / Logistic Regression on features that are
available **before** the race (grid, qualifying, prior-race averages, prior
championship points, recent form) using a time-based train/test split.

In [ ]:
from app.mining.prediction import run_prediction

with SessionLocal() as db:
    pred = run_prediction(db)

print('split:', pred['split'])
print('train:', pred['train_size'], '| test:', pred['test_size'],
      '| base rate:', pred['base_rate'])
for name, m in pred['models'].items():
    print(f'  {name:>18}: acc={m["accuracy"]:.3f} '
          f'f1={m["f1"]:.3f} auc={m["roc_auc"]}')

Top features that drive the Random Forest prediction:

In [ ]:
pd.DataFrame(pred['feature_importances'])

## 3. Association rules (Apriori)

One transaction per driver-race entry; items are derived bins (qualifying/grid
buckets, constructor tier, pit stops, outcome, position change, reliability,
pace). Rules are filtered by support / confidence / lift.

In [ ]:
from app.mining.association import run_association

with SessionLocal() as db:
    assoc = run_association(db)

print('transactions =', assoc['dataset_size'],
      '| items =', assoc['item_count'], '| rules =', assoc['rule_count'])
pd.DataFrame(assoc['rules']).head(10)

## 4. Performance classification (E/S/A/P)

A documented composite score — 60% track position + 40% championship value —
maps every race result (and every driver career) into EXCELLENT / STRONG /
AVERAGE / POOR.

In [ ]:
from app.mining.classification import run_classification

with SessionLocal() as db:
    cls = run_classification(db)

print('formula:', cls['formula'])
pd.DataFrame(cls['distribution']).T

Top driver careers by mean performance score (min 10 scored races):

In [ ]:
pd.DataFrame(cls['driver_classification']).head(10)